In [ ]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from typing import Annotated, Dict



class GetDeviceInFovInput(BaseModel):
    isInFov: bool = Field(
        default=True,
        description= """
    - isInFov: If True, returns devices that are within the user's line of sight.
    """
    )
    order: str = Field(
        default="proximity",
        description= """
    - order : The sorting method for devices. Possible values are:
          - "proximity" (closest first, default)
          - "right" (when you need to sort the devices from right to left)
          - "high" (when you need to sort the devices from high to low with  z value)
    """
    )
    range: Optional[float] = Field(
        default=None,
        description="""
    - range : The distance range from user to search for devices (in meters). 
          Optional, default is no range limitation.
    """
    )

    class Config:
        extra = "forbid"  # additionalProperties: false にする





@tool
def getDeviceInFov(params: GetDeviceInFovInput) -> Dict:
    """
    This function retrieves devices that are within the user's line of sight.
    return value is a list of devices in which its order is according to the argument.
    """
    
    try:
        print()
        print("=====================[TOOL] getDevicesInSights================")
        request_body = {
            "isInFov": param.isInFov,
            "order": param.order,
            "range": param.range
            }
        
        print(f"Sending POST request to {base_url}/fov with body: {request_body}")
        response = httpx.post(f"{base_url}/fov", json=request_body)
        
        if response.status_code == 200:
            response_data = response.json()
            received_devices = response_data["devices"]
            if response_data.get("status") == "success":
                print(response_data)
                print("================================================")
                print()
                return response_data
            else:
                return {"status": "error", "message": "Server responded with an error", "details": response_data}
        else:
            return {"status": "error", "message": f"HTTP Error {response.status_code}", "details": response.text}
    except Exception as e:
        return {"status": "error", "message": str(e)}

In [ ]:
class GetDeviceInDirectionInput(BaseModel):
    direction: str = Field(
        default="Front",
        description=  """
    - direction (str): The direction to search for devices. Possible values are:
          - "Front" (devices in front of the user)
          - "Back" (devices behind the user)
          - "Left" (devices to the left of the user)
          - "Right" (devices to the right of the user)
    """
    )
    order: str = Field(
        default="proximity",
        description= """
    - order : The sorting method for devices. Possible values are:
          - "proximity" (closest first, default)
          - "right" (when you need to sort the devices from right to left)
          - "high" (when you need to sort the devices from high to low with  z value)
    """
    )
    range: Optional[float] = Field(
        default=None,
        description="""
    - range : The distance range from user to search for devices (in meters). 
          Optional, default is no range limitation.
    """
    )
@tool
def getDeviceInDirection(
    param : GetDeviceInDirectionInput
) -> Dict:
    """
    This function retrieves devices based on the user's retrieved direction.
    return boolean value indicates whether the devices are successfully received.
    """
    try:
        print()
        print("=====================[TOOL] getDevicesInSights================")
        request_body = {
        "direction" : param.direction,
        "order": param.order ,
        "range" : param.range
    }
        
        
        print(f"Sending POST request to {base_url}/direction with body: {request_body}")
        response = httpx.post(f"{base_url}/direction", json=request_body)
        
        if response.status_code == 200:
            response_data = response.json()
            received_devices = response_data["devices"]
            if response_data.get("status") == "success":
                print(response_data)
                
                print("================================================")
                print()
                return response_data
            else:
                return {"status": "error", "message": "Server responded with an error", "details": response_data}
        else:
            return {"status": "error", "message": f"HTTP Error {response.status_code}", "details": response.text}
    except Exception as e:
        return {"status": "error", "message": str(e)}

In [ ]:


@tool
def getDevices(

    order: Annotated[str, """
    - order (str): The sorting method for devices. Possible values are:
          - "proximity" (closest first, default)
          - "right" (when you need to sort the devices from right to left)
          - "high" (when you need to sort the devices from high to low with z value)
    """] = "proximity",
    range: Annotated[float, """
    - range (float): The distance range from user to search for devices (in meters). 
          Optional, default is no range limitation.
    """] = 0.0,
) -> Dict:
    """
    This function retrieves devices based on the user's retrieved direction.
    return boolean value indicates whether the devices are successfully received.
    """
    try:
        print()
        print("=====================[TOOL] getDevices================")
        request_body = {
        "order": order ,
     
    }
        
        if range != 0: 
            request_body["range"] = range
        
        print(f"Sending POST request to {base_url}/all with body: {request_body}")
        response = httpx.post(f"{base_url}/all", json=request_body)
        
        if response.status_code == 200:
            response_data = response.json()
            received_devices = response_data["devices"]
            if response_data.get("status") == "success":
                print(response_data)
                
                print("================================================")
                print()
                return response_data
            else:
                return {"status": "error", "message": "Server responded with an error", "details": response_data}
        else:
            return {"status": "error", "message": f"HTTP Error {response.status_code}", "details": response.text}
    except Exception as e:
        return {"status": "error", "message": str(e)}

In [4]:
import httpx




url = "http://127.0.0.1:8800/llm_agent"

data = {
"llm_message" : "目の前の電気を点けてください", 
    "task_id" : "1"
}



res = httpx.post(url, json=data, timeout=60)

In [6]:
res.content

b'{"output":"It seems like there are no devices currently identified as being \\"in front of you\\" according to the available data. Please ensure that the devices are correctly aligned with your field of view or provide more specific instructions so I can assist you better."}'

In [1]:
import httpx

url = "http://127.0.0.1:7070/furniture/get"

data = {"furnitureType":"TV", "range":3}
operator = httpx.post(url=url, json=data)

ConnectError: [WinError 10061] 対象のコンピューターによって拒否されたため、接続できませんでした。

In [2]:
import httpx

url = "http://127.0.0.1:7070/device/direction"

data = {"direction":"Up", "order" : "proximity" , "range":3}
operator = httpx.post(url=url, json=data)
import json 
print(json.loads(operator.content))

{'status': 'success', 'devices': [{'id': 'd4dadccf-1ea0-4bef-b570-a62d5b8e1d0a', 'name': '壁ランプ１', 'position': {'x': 0.395174861, 'y': 1.21728611, 'z': -1.40179849}, 'distance_from_user': 1.89815366}, {'id': '8a0f3d8a-5ef8-4d72-8f4e-959588533b04', 'name': '壁ランプ３', 'position': {'x': -1.21962619, 'y': 1.21423137, 'z': -1.458765}, 'distance_from_user': 2.25606775}, {'id': 'aae4e77d-4d1b-4f9c-8db3-a83b6079ac56', 'name': '天井ライト2', 'position': {'x': -0.15005669, 'y': 2.225976, 'z': -0.517544448}, 'distance_from_user': 2.29027}, {'id': '688c5282-ec03-4ca9-8060-a6cae2c85834', 'name': '天井ライト5', 'position': {'x': -0.220940217, 'y': 2.23674273, 'z': 1.49117672}, 'distance_from_user': 2.69730234}, {'id': '75e4bc21-50ad-4d9d-ba56-2c908c6276c1', 'name': '壁ランプ２', 'position': {'x': 2.03634238, 'y': 1.2203908, 'z': -1.34390175}, 'distance_from_user': 2.72802424}, {'id': '91087252-3fe6-4e8c-8b6b-47618b86bfe8', 'name': '天井ライト1', 'position': {'x': 1.88667381, 'y': 2.22982883, 'z': -0.445693165}, 'distance_

In [28]:
import httpx
import json
url = "http://127.0.0.1:7070/furniture/get"

data = {"furnitureType": "SHELF", "range": 4}
print(data)
operator = httpx.post(url=url, json=data)
json.loads(operator.content)

{'furnitureType': 'SHELF', 'range': 4}


{'status': 'success',
 'devices': [{'id': '982c8f63-7701-498d-82d5-4d4ec9c91888',
   'name': '棚ライト８',
   'position': {'x': 0.404444754, 'y': 1.145, 'z': -0.1622875},
   'distance_from_furniture': 1.2251277},
  {'id': '623228c0-4b86-484c-82a4-463f90fcf803',
   'name': '棚ライト５',
   'position': {'x': 0.337384552, 'y': 1.852, 'z': -0.1287033},
   'distance_from_furniture': 1.88687491},
  {'id': '92112672-26fa-4d7a-a5d5-66534d52d76a',
   'name': '棚ライト９',
   'position': {'x': 1.4219743, 'y': 1.145, 'z': -0.6718732},
   'distance_from_furniture': 1.94536614},
  {'id': '3c923e98-a278-4283-838a-9f161456f8f1',
   'name': '棚ライト4',
   'position': {'x': -0.995776057, 'y': 1.852, 'z': 0.5389526},
   'distance_from_furniture': 2.17070127},
  {'id': '0a16141b-e193-4056-9d33-b6f74f6a58a8',
   'name': '棚ライト６',
   'position': {'x': 1.10723782, 'y': 1.852, 'z': -0.514251053},
   'distance_from_furniture': 2.21818256},
  {'id': '46a35f91-6b89-486b-afda-f12e9ce4e555',
   'name': '棚ライト７',
   'position': {'x':

In [10]:
operator.content


b'\xef\xbb\xbf{"status":"success","devices":[{"id":"855b94e8-74c8-4e1f-b059-cfd4de49e7ed","name":"\xe6\xa4\x8d\xe7\x89\xa9\xe3\x83\xa9\xe3\x82\xa4\xe3\x83\x88\xef\xbc\x93","position":{"x":-2.38418579E-07,"y":0.0,"z":1.65300012},"distance_from_user":1.65299988},{"id":"405e2bf8-dfa5-425c-a17f-9a78681f65b4","name":"\xe3\x83\x95\xe3\x83\xad\xe3\x82\xa2\xe3\x83\xa9\xe3\x82\xa4\xe3\x83\x88\xef\xbc\x91","position":{"x":-1.00000036,"y":0.0,"z":1.653},"distance_from_user":1.93194425},{"id":"74b6942c-b0d7-4fa5-affc-8b94a9cadb2d","name":"\xe3\x83\x95\xe3\x83\xad\xe3\x82\xa2\xe3\x83\xa9\xe3\x82\xa4\xe3\x83\x88\xef\xbc\x92","position":{"x":0.9999999,"y":0.0,"z":1.65300024},"distance_from_user":1.93194425},{"id":"a611f12e-a7c6-42b7-afa3-dc6857117278","name":"\xe3\x83\x86\xe3\x83\xbc\xe3\x83\x96\xe3\x83\xab\xe3\x83\xa9\xe3\x82\xa4\xe3\x83\x881","position":{"x":-1.44900048,"y":0.77,"z":4.974591},"distance_from_user":5.23822975},{"id":"f12fa748-495a-43b4-bf45-ea6076ec9d4b","name":"\xe3\x83\x86\xe3\x83\

In [2]:
import httpx
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from typing import Annotated, Dict
import os 
from dotenv import load_dotenv
import os

# .envファイルの絶対パスを指定
env_path = os.path.join("..", ".env")  # 親ディレクトリの.envを指定
load_dotenv(env_path)
base_url = os.getenv("XR_SERVER_API")




@tool
def getDeviceInFov(
    isInFov: Annotated[bool, """- isInFov (bool): If True, returns devices that are within the user's line of sight. Default: True"""],
    order: Annotated[str, """- order (str): Sorting method for devices. Possible values: "proximity", "right", "high". Default: proximity"""],
    range: Annotated[float, """- range (float): Distance range (meters). Input 0.0 if specification is not required."""] 
) -> Dict:
    """
    This function retrieves devices that are within the user's line of sight.
    return boolean value indicates whether the devices are successfully received in back side.
    """
    
    try:
        # print("\n[TOOL FOV] getDevicesInFov")
        
        request_body = {
            "isInFov": isInFov,
            "order": order
        }

        if range != 0.0:
            request_body["range"] = range

        # print(f"Sending POST request to {base_url}/fov with body: {request_body}")
        response = httpx.post(f"{base_url}/device/fov", json=request_body)
        print("Received Devices")

        if response.status_code == 200:
            response_data = response.json()

            # `param` が存在しない場合は作成する
            response_data.setdefault("param", {})

            # `param` に値を追加
            response_data["param"]["filter_type"] = "fov"
            response_data["param"]["isInFov"] = isInFov
            response_data["param"]["order"] = order
            if range != 0.0:
                response_data["param"]["range"] = range

            if response_data.get("status") == "success":
                # print(f"[TOOL FOV] {response_data}")
                return response_data
            else:
                return {"status": "error", "message": "Server responded with an error", "details": response_data}
        else:
            return {"status": "error", "message": f"HTTP Error {response.status_code}", "details": response.text}

    except Exception as e:
        return {"status": "error", "message": str(e)}

In [5]:
getDeviceInFov({"isInFov" : True, "order" : "proximity", "range" : 10})

Received Devices


{'status': 'success',
 'devices': [{'id': 'c04bfeba-ef52-4bb1-ab9a-a9ee160356f7',
   'name': 'スイッチボット(Clone)',
   'position': {'x': 0.127027035, 'y': -0.422706246, 'z': 0.6196184},
   'distance_from_user': 0.760751843},
  {'id': '3d4eda69-61a9-4e31-8045-9ed537651661',
   'name': 'スイッチボット(Clone)',
   'position': {'x': 0.224137992, 'y': -0.42476812, 'z': 0.601885},
   'distance_from_user': 0.7700203},
  {'id': 'e91e3b96-e4b5-4549-ae4f-7b4cc28f2f6a',
   'name': 'スイッチボット(Clone)',
   'position': {'x': 0.346939325, 'y': -0.426159471, 'z': 0.588372946},
   'distance_from_user': 0.8050847}],
 'param': {'filter_type': 'fov',
  'isInFov': True,
  'order': 'proximity',
  'range': 10.0}}